## Reservoir Computing and Echo State Networks

Reservoir Computing (RC) is an RNN framework designed to model temporal correlations in data. A defining feature of this architecture is that the recurrent part is not trained, which significantly reduces computational costs and training time compared to traditional RNNs.

An RC model is generally composed of three entities:
1. **Input Layer**
2. **Reservoir**
3. **Output Layer** (Readout)

Assuming the most general case, we consider an input $x_t \in \mathbb{R}^K$ and an output $y_t \in \mathbb{R}^L$. 
The core principle of RC is to map the input data into a high-dimensional embedding space (the reservoir, with dimension $N$, where $N \gg K$) to capture complex temporal dependencies without explicitly modeling the underlying dynamics. The final prediction is performed by the readout layer, which maps the reservoir state back to the output space.

In an Echo State Network (ESN), the specific RC model used in this project, the embedding mapping is purely deterministic and remains untrained. The system is defined by the following equations:

$$
s_{t+1} = f(W_{in}x_{t+1} + Ws_t)
$$
$$
y_t = g(Rs_t)
$$

Where the weight matrices are initialized as follows:

* **$W_{in} \in \mathbb{R}^{N \times K}$**: The input weight matrix, which is **fixed** and initialized with values drawn from a **Uniform distribution** $\mathcal{U}(-1, 1)$.
* **$W \in \mathbb{R}^{N \times N}$**: The reservoir weight matrix, which is **fixed** and initialized using **Gaussian random variables** with mean $\mu = 0$ and variance $\sigma^2 = 1$. To ensure the **Echo State Property (ESP)**, $W$ is subsequently scaled such that its spectral radius $\rho(W) < 1$.
* **$R \in \mathbb{R}^{L \times N}$**: The readout weight matrix, representing the **only trainable** part of the network.
* **$s_t \in \mathbb{R}^N$**: The reservoir state vector at time $t$.
* **$f$** and **$g$**: Activation functions (typically *tanh* for the reservoir and *identity* for the output).

### Our Project Objectives

Standard ESNs are limited by their deterministic nature, providing only point estimates. Our objective is to implement probabilistic readout frameworks to quantify forecast uncertainty:

* **MCMC & SVI**: To perform Bayesian inference on the readout weights.
* **SSVS**: To implement sparsity and select the most relevant reservoir nodes.
* **BNN**: To model the output through a Bayesian Neural Network approach.
* **Quantile Regression**: To estimate specific quantiles of the predictive distribution.

## Stochastic Variational Inference (SVI)

Stochastic Variational Inference (SVI) is a Bayesian modeling framework that approximates the intractable posterior distribution $p(R | \mathcal{D})$ of the readout weights by introducing a simpler, parameterized distribution $q_\phi(R)$, known as the **variational distribution**. 

The goal is to find the parameters $\phi$ that make $q_\phi(R)$ as close as possible to the true posterior. This is achieved by minimizing the **Kullback-Leibler (KL) divergence** between the two distributions:

$$
\min_\phi \text{KL}(q_\phi(R) || p(R | \mathcal{D}))
$$

Since the true posterior is unknown, this minimization is equivalent to maximizing the **Evidence Lower Bound (ELBO)**. The ELBO loss function is defined as:

$$
\text{ELBO}(\phi) = \mathbb{E}_{q_\phi(R)}[\log p(\mathcal{D} | R)] - \text{KL}(q_\phi(R) || p(R))
$$

Where:
1.  **Likelihood Term** ($\mathbb{E}_{q_\phi(R)}[\log p(\mathcal{D} | R)]$): Measures how well the model fits the data given the reservoir states.
2.  **Complexity/Regularization Term** ($\text{KL}(q_\phi(R) || p(R))$): Penalizes the variational distribution for deviating too much from the prior $p(R)$, acting as a natural regularizer.

SVI leverages **stochastic optimization** to maximize the ELBO, making it significantly more scalable to large datasets than traditional MCMC methods. In our project, we use SVI to obtain a probabilistic readout $R$ that provides not just a prediction, but a full distribution of possible outcomes, allowing also for **uncertainty quantification**.

## Implementation Details

To bridge the gap between theory and practice, we implemented the ESN Readout using the **Pyro** probabilistic programming library. The model is structured to perform Bayesian inference on the readout weights while maintaining computational efficiency.

### The Generative Model (`model_fn`)
The model defines how the data is generated from the latent variables.

* **Prior on $R$**: We sample the readout weights $R$ from a standard Normal distribution $\mathcal{N}(0, 1)$. We use `.to_event(2)` to ensure that Pyro treats $R$ as a single multidimensional multidimensional block (a matrix of size $N \times 119$) rather than independent scalars.
* **Linear Projection**: The mean of our prediction, $\mu$, is calculated as the direct product between the reservoir states and the weights $R$. This confirms the use of an **identity activation function** for the output.
* **Likelihood**: We assume the observations follow a Normal distribution centered at $\mu$ with a noise scale $\sigma$. The parameter $\sigma$ is sampled from a `HalfNormal` distribution to ensure positivity.

### The Variational Guide (`AutoLowRankMultivariateNormal`)
Since the true posterior $p(R | \mathcal{D})$ is intractable due to the high dimensionality of the reservoir ($N$), we use an automated variational guide.

* **Low-Rank Approximation**: We employ `AutoLowRankMultivariateNormal`. This choice is critical: instead of estimating a full $N \times N$ covariance matrix, we approximate it using a low-rank structure $\Sigma = D + VV^T$.
* **Rank Selection**: We set the rank of the approximation to $\text{RANK} = \sqrt{N}$. This heuristic effectively balances the need to capture correlations between neurons while keeping the number of trainable parameters manageable.

### Training Objective: Negative ELBO
The model is trained using the **SVI** (Stochastic Variational Inference) engine. In line with PyTorch's optimization standards, we minimize the **Negative ELBO Loss**:

$$
\text{Loss}_{SVI} = - \left( \mathbb{E}_{q_\phi(R)}[\log p(\mathcal{D} | R)] - \text{KL}(q_\phi(R) || p(R)) \right)
$$

As the loss decreases during training, the Evidence Lower Bound (ELBO) increases, signifying that our variational distribution $q_\phi(R)$ is successfully converging toward the true posterior of the readout weights.